## Architecture reference for this lab

**Step 11 — Task/Tool Agent with AgentCore Gateway**

![Step 11 — Task/Tool Agent with AgentCore Gateway](images/step-11-task-tool-gateway.png)



# Lab 3 — Task/Tool Agent via AgentCore Gateway

**What this lab is.** We give CareConnect the ability to *do things* (not just answer
questions) through three **synthetic** hospital tools: check an appointment, check refill
eligibility, and *stage* (prepare, but not submit) a prescription refill. These are exposed
through an **AgentCore Gateway**, which turns our AWS Lambda function into "tools" the agent
can call.

**Why we do it.** A useful patient assistant sometimes needs to take an action, like starting
a refill. But high-impact actions must never happen automatically. So the refill tool only
*stages* the request (marks it "awaiting approval, not submitted") for a human to approve.

**Why it's needed here.** This demonstrates the "human-in-the-loop" safety principle: the
assistant can prepare an action, but a person must approve it. In healthcare that boundary is
essential.

**How it helps the project.** The Supervisor can call these tools when appropriate. The
Gateway is the standard, secure way to give an agent tools.

**The use case.** "I need a refill of my medication" → the agent checks eligibility and
stages the refill, clearly telling the patient it is *awaiting approval*, never claiming it
was submitted.

> **Note on the data:** everything here is fake/synthetic. No real hospital system is touched.

---

## Prerequisite — run these two setup cells first (every lab has them)

Before this lab's own steps, run the **two setup cells** below. Every notebook (lab-00
through lab-08) starts with these same two cells — on purpose, not by mistake.

**Why they repeat in every notebook.** Each notebook runs in its own fresh "kernel"
(a separate Python session) with no memory of the other notebooks. So each one has to set
itself up from scratch. These two cells are that setup.

**Cell 1 — "bootstrap":** finds the project's main folder (the one containing `lab_helpers`)
no matter where the notebook is opened from, so `import lab_helpers...` always works.

**Cell 2 — "preflight":** checks all helper files are present before the lab begins, and stops
with a clear message if anything is missing — instead of failing confusingly later.

**Do I run them?** Yes — run both, in order, at the top of **every** lab. They take a second
and prevent the most common setup problems. After these two, continue with the lab's steps.

In [1]:
# WHAT THIS CELL DOES (plain English):
# - It looks at the current folder, then its parent, then its parent's parent, and so on,
#   until it finds the folder that contains "lab_helpers". That folder is our project root.
# - It then switches into that folder and adds it to Python's search path, so that
#   'import lab_helpers...' works from anywhere.
# - If it never finds "lab_helpers", it stops with a clear message instead of a confusing error later.
# You do not need to edit anything here — just run it first.
# === CareConnect bootstrap — run me first ===
# Makes this notebook work from any folder and gives a clear error if the
# lab_helpers package is missing (e.g. not uploaded to the repo).
import os, sys

def _find_repo_root(start=None):
    here = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(here, "lab_helpers")):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            return None
        here = parent

_root = _find_repo_root()
if _root is None:
    raise RuntimeError(
        "Could not find the 'lab_helpers/' folder from " + os.getcwd() + ".\n"
        "This means the helper package is not next to the notebooks.\n"
        "Fix: make sure lab_helpers/ and requirements.txt are in the same folder\n"
        "as these .ipynb files (see README > Setup). Then re-run this cell.")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Repo root:", _root)

Repo root: /home/sagemaker-user/careconnect-patient-assistant-k21


In [4]:
# WHAT THIS CELL DOES (plain English):
# - It checks that each helper file we rely on is actually present on disk.
# - If any are missing, it stops now with a clear list of what to upload, rather than
#   failing in the middle of the lab.
# Run this straight after the bootstrap cell above.
# === Preflight: confirm every helper file is present BEFORE running the lab ===
import os
_required = [
    "requirements.txt",
    "lab_helpers/__init__.py",
    "lab_helpers/utils.py",
    "lab_helpers/careconnect_agents.py",
    "lab_helpers/deterministic_safety.py",
    "lab_helpers/runtime_entrypoint.py",
]
_missing = [f for f in _required if not os.path.isfile(f)]
if _missing:
    raise RuntimeError("Missing required files:\n  - " + "\n  - ".join(_missing) +
        "\n\nUpload the full lab_helpers/ folder + requirements.txt, then re-run.")
print("Preflight OK — all helper files present.")

Preflight OK — all helper files present.


### Step 1 — Create the mock hospital tools (an AWS Lambda function)

**What:** create a small AWS **Lambda** function (code that runs in the cloud on demand) that
pretends to be the hospital's systems and returns fake appointment/refill data.

**Why:** we don't have (or want) access to a real hospital system in a lab. This mock stands
in for it so we can demonstrate the tool-calling and human-approval flow safely.

In [5]:
# WHAT THIS CELL DOES (plain English):
# - Defines a small cloud function (AWS Lambda) with three fake hospital operations:
#     * check_appointment_status  -> returns a fake appointment
#     * check_refill_eligibility  -> returns fake "eligible: true"
#     * stage_refill_request      -> prepares a refill but marks submitted=False (NOT sent)
# - Zips that code up in memory, creates the permission role it needs, and uploads it as a
#   Lambda function. If it already exists, it just reuses it.
import boto3, json, zipfile, io, time
import lab_helpers.utils as u

lambda_client = boto3.client("lambda", region_name=u.REGION)
account = u.get_aws_account_id()

LAMBDA_SRC = '''
import uuid
def get_tool_name(event, context):
    try:
        custom = context.client_context.custom
        full = custom.get("bedrockAgentCoreToolName", "")
        return full.split("___", 1)[1] if "___" in full else full
    except Exception:
        return event.get("_tool_name", "")
def lambda_handler(event, context):
    t = get_tool_name(event, context)
    if t == "check_appointment_status":
        return {"patient_reference": event.get("patient_reference","PATIENT-DEMO-001"),
                "appointment_status":"scheduled","department":"Gastroenterology",
                "appointment_reference":"APT-DEMO-1001"}
    if t == "check_refill_eligibility":
        return {"drug": event.get("drug","Unknown"),"eligible":True,
                "refills_remaining":2,"data_type":"synthetic"}
    if t == "stage_refill_request":
        return {"action":"submit_refill","drug":event.get("drug","Unknown"),
                "status":"awaiting_approval",
                "idempotency_key":event.get("idempotency_key",str(uuid.uuid4())),
                "submitted":False}
    return {"error":"Unknown tool","tool_name":t}
'''

# Zip in-memory
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as z:
    z.writestr("lambda_function.py", LAMBDA_SRC)
buf.seek(0)

# Lambda needs an execution role — reuse a simple basic-exec role.
lambda_role_arn = u._create_role(
    u.name("CareConnectMockToolsRole"), "lambda.amazonaws.com",
    {"Version":"2012-10-17","Statement":[{"Effect":"Allow",
       "Action":["logs:CreateLogGroup","logs:CreateLogStream","logs:PutLogEvents"],
       "Resource":"*"}]},
    u.name("CareConnectMockToolsPolicy"))

try:
    fn = lambda_client.create_function(
        FunctionName=u.MOCK_TOOLS_LAMBDA, Runtime="python3.12",
        Role=lambda_role_arn, Handler="lambda_function.lambda_handler",
        Code={"ZipFile": buf.read()}, Timeout=30)
    print("Created Lambda:", fn["FunctionArn"])
except lambda_client.exceptions.ResourceConflictException:
    fn = lambda_client.get_function(FunctionName=u.MOCK_TOOLS_LAMBDA)["Configuration"]
    print("Reusing Lambda:", fn["FunctionArn"])
lambda_arn = fn["FunctionArn"]

Reusing existing role CareConnectMockToolsRole-sdk
Created Lambda: arn:aws:lambda:us-east-1:831963379350:function:careconnect-mock-hospital-tools-sdk


### Step 2 — Publish the tools through an AgentCore Gateway

**What:** create an **AgentCore Gateway** and attach our Lambda to it, described as three
tools with clear input schemas.

**Why:** a Gateway is the secure doorway between an agent and its tools. It turns our Lambda
into standard "MCP tools" that any agent can discover and call. (For this lab we use no inbound
auth to keep it simple; a real deployment would require authentication.)

In [6]:
# WHAT THIS CELL DOES (plain English):
# - Creates the Gateway (the secure doorway to our tools) and the permission role it uses to
#   call our Lambda.
# - Saves the Gateway's ID and URL into Parameter Store.
# - NOTE: 'authorizerType=NONE' means no login is required to reach the gateway — fine for a
#   synthetic lab, but a real system would use JWT/IAM authentication here.
acc = boto3.client("bedrock-agentcore-control", region_name=u.REGION)

gw_role_arn = u._create_role(
    u.name("CareConnectGatewayRole"), "bedrock-agentcore.amazonaws.com",
    {"Version":"2012-10-17","Statement":[{"Effect":"Allow",
       "Action":["lambda:InvokeFunction"],"Resource": lambda_arn}]},
    u.name("CareConnectGatewayPolicy"))

# NOTE: adjust create_gateway args to your installed bedrock-agentcore-control schema.
gw = acc.create_gateway(
    name=u.GATEWAY_NAME,
    roleArn=gw_role_arn,
    protocolType="MCP",
    authorizerType="NONE",
    description="CareConnect synthetic hospital tools (SDK build).")
gateway_id = gw["gatewayId"]
gateway_url = gw["gatewayUrl"]
u.put_ssm_parameter(f"{u.SSM_PREFIX}/gateway_id", gateway_id)
u.put_ssm_parameter(f"{u.SSM_PREFIX}/gateway_url", gateway_url)
print("Gateway:", gateway_id, gateway_url)

Created role CareConnectGatewayRole-sdk
Gateway: careconnect-tools-gateway-sdk-jpjeagnibi https://careconnect-tools-gateway-sdk-jpjeagnibi.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp


In [7]:
# WHAT THIS CELL DOES (plain English):
# - Describes each of the three tools to the Gateway: its name, what it does, and what inputs
#   it expects. This is how the agent knows which tools exist and how to call them.
# - Then attaches our Lambda to the Gateway as the thing that actually runs those tools.
tool_schema = [
  {"name":"check_appointment_status",
   "description":"Check the status of a synthetic Riverside Health appointment.",
   "inputSchema":{"type":"object","properties":{
      "patient_reference":{"type":"string","description":"Synthetic patient reference."}},
      "required":["patient_reference"]}},
  {"name":"check_refill_eligibility",
   "description":"Check synthetic prescription refill eligibility. Does not submit a refill.",
   "inputSchema":{"type":"object","properties":{
      "drug":{"type":"string","description":"Synthetic medication name."}},
      "required":["drug"]}},
  {"name":"stage_refill_request",
   "description":"Prepare a synthetic refill request for human approval. Does not submit.",
   "inputSchema":{"type":"object","properties":{
      "drug":{"type":"string","description":"Synthetic medication name."},
      "idempotency_key":{"type":"string","description":"Unique id to prevent duplicates."}},
      "required":["drug","idempotency_key"]}},
]

acc.create_gateway_target(
    gatewayIdentifier=gateway_id,
    name=u.name("careconnect-hospital-tools"),
    targetConfiguration={"mcp":{"lambda":{
        "lambdaArn": lambda_arn,
        "toolSchema":{"inlinePayload": tool_schema}}}},
    credentialProviderConfigurations=[{"credentialProviderType":"GATEWAY_IAM_ROLE"}])
print("Target attached.")

Target attached.


### Step 3 — The agent uses the tools

**What:** connect an agent to the Gateway, let it discover the tools, and ask it to stage a
refill.

**Why:** this proves the whole chain works — the agent finds the tools, calls the right one,
and (crucially) reports the refill as *awaiting approval, not submitted*, exactly as required.

In [8]:
# WHAT THIS CELL DOES (plain English):
# - Connects to the Gateway and asks it to list the available tools.
# - Builds a Task/Tool Agent with strict rules (never diagnose, never claim a refill was
#   submitted) and gives it those tools.
# - Asks it to stage a refill; the output shows it staged the request and left it AWAITING
#   APPROVAL (submitted=false) — the human-in-the-loop safeguard working.
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

TASK_PROMPT = '''You are the CareConnect Task and Tool Agent for Riverside Health.
You may check appointment status, check refill eligibility, and stage a refill for
human approval. Never diagnose, never recommend medication or dosage, never claim a
staged refill was submitted. A staged refill stays awaiting_approval / submitted=false.
All data is synthetic.'''

mcp_client = MCPClient(lambda: streamablehttp_client(url=gateway_url))
with mcp_client:
    tools = mcp_client.list_tools_sync()
    agent = Agent(model=BedrockModel(model_id=u.MODEL_ID),
                  system_prompt=TASK_PROMPT, tools=tools)
    print(agent("Prepare a refill request for DemoMedication using key REFILL-DEMO-001."))


Tool #1: careconnect-hospital-tools-sdk___stage_refill_request
Your refill request for DemoMedication has been staged and is awaiting approval. The idempotency key REFILL-DEMO-001 was used, and the request has not been submitted yet.Your refill request for DemoMedication has been staged and is awaiting approval. The idempotency key REFILL-DEMO-001 was used, and the request has not been submitted yet.



## Lab 3 complete ✅

Three synthetic tools discoverable/callable via Gateway; refills only stage.